In [1]:
# this notebook will merge marker.json into ohlcv.json
# then send to visualize on TVJS3
source = '../../data/bigData/BTCUSD-15m.json'
marker = '../../data/bigData/15m-overlay-v3.json'

In [2]:
import json
import math
import os

def fillnull(obj):
    """
    Python None -> JSON null by default
    """
    if isinstance(obj, list):
        return [fillnull(v) for v in obj]
    elif isinstance(obj, dict):
        return {k: fillnull(v) for k, v in obj.items()}
    elif obj is None:
        return None
    elif isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    elif obj in ("", "NaN", "null", "Null", "NULL"):
        return None
    return obj


def fill_nan_null(obj):
    if isinstance(obj, list):
        return [fill_nan_null(v) for v in obj]
    elif isinstance(obj, dict):
        return {k: fill_nan_null(v) for k, v in obj.items()}
    elif obj is None:
        return 0
    elif isinstance(obj, float) and math.isnan(obj):
        return 0
    return obj

def merge_data(source_path, marker_path, out_path=None):
    with open(source_path) as f:
        source = json.load(f, parse_constant=lambda x: float('nan') if x == 'NaN' else None)

    with open(marker_path) as f:
        marker = json.load(f, parse_constant=lambda x: float('nan') if x == 'NaN' else None)

    source.pop("onchart", None)
    source["onchart"] = fillnull(marker.get("onchart", []))
    # source["onchart"] = marker.get("onchart", [])

    if "offchart" in marker:
        source.pop("offchart", None)
        # source["offchart"] = fill_nan_null(marker["offchart"])
        # source["offchart"] = marker["offchart"]

    if out_path:
        with open(out_path, "w") as f:
            json.dump(source, f)
        print(f"Saved merged data to {out_path}")

    return source

# build out_path: same filename as source but with '-marked' and saved to data/bigData
_src_name = os.path.splitext(os.path.basename(source))[0]
_out_path = f'../../data/bigData/{_src_name}-v3-null.json'

# execute
result = merge_data(source, marker, out_path=_out_path)


Saved merged data to ../../data/bigData/BTCUSD-15m-v3-null.json


In [ ]:
# symlink to